In [8]:
import warnings
warnings.simplefilter("ignore", FutureWarning)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

# sklearn
# SVM
from sklearn.svm import SVC

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.experimental import enable_halving_search_cv 

from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, cross_val_score,
    RandomizedSearchCV, StratifiedKFold,train_test_split,HalvingRandomSearchCV
)

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             average_precision_score,roc_curve)


from sklearn.base import BaseEstimator, TransformerMixin, clone
from scipy.stats import randint, uniform, loguniform

# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.preprocess_utils_diab3a import * #(NOVO atualizações)

print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))
start_inicial = time.time()


#Processo iniciado em: 17:29:26


## 1. Load Data & Pipeline

In [5]:
BASE = Path.cwd().parent

PP3a = joblib.load(BASE/'src'/'preprocess_diabetes_v3a.joblib')['preprocessador']

DATA_DIR = BASE/"data"/"raw"
X_train = pd.read_csv(DATA_DIR/"X_train_raw.csv")
X_test  = pd.read_csv(DATA_DIR/"X_test_raw.csv")
y_train = pd.read_csv(DATA_DIR/"y_train_raw.csv").values.ravel()
y_test  = pd.read_csv(DATA_DIR/"y_test_raw.csv").values.ravel()

mtd_scoring='roc_auc'
print("\n#Processo em:", time.strftime("%H:%M:%S"))


#Processo em: 17:12:29


## 2.Baseline

In [7]:
print("#Processo iniciado em:", time.strftime("%H:%M:%S"))

# MODEL BASE
#model_base = SVC(kernel='rbf', random_state=42, probability=True)

# dual=False é preferível quando n_samples > n_features
model_base = LinearSVC(dual=False, C=1.0, random_state=42, max_iter=2000)
calibrated_svc = CalibratedClassifierCV(estimator=model_base, cv=3, method='sigmoid')
# modelo precisa que os dados sejam numericos e normalizados.
# o preprocessador pp3a não faz isso(ainda). portanto faremos aqui.
cat_cols = ['gender', 'ethnicity', 'smoking_status', 'employment_status', 'Age_Group', 'PAW_Group']

svm_one_hot= ColumnTransformer([
    ('cat_to_nums', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
], remainder='passthrough')

pipe_base = Pipeline([
    ('preprocess_v3a', PP3a),                 # PP3a
    ('ohe_transform', svm_one_hot), # Converte as 6 categorias em números
    ('scaler', StandardScaler()),            # Normaliza todas as features todas as features
    ('model', calibrated_svc)
])

print(f"# Iniciando treino rápido em {X_train.shape[0]} linhas...")
pipe_base.fit(X_train, y_train)
print("✅ Treino concluído!")
# =====================================================
# 1) Predição
# =====================================================
y_pred_base=pipe_base.predict(X_test)


# =====================================================
# 2) Otimização do threshold de decisão
# =====================================================
# Como classificadores probabilísticos usam threshold padrão de 0.5,
# testamos diferentes valores para verificar se existe um ponto que
# maximize a acurácia no conjunto de teste.
best_th_base,max_acc_base,y_probs_base=best_threshold2(pipe_base, X_train, y_train,X_test,y_test)

# =====================================================
# 3) Desempenho em validação cruzada
# =====================================================
# A validação cruzada utiliza a mesma métrica definida
# no processo de tuning para avaliar a estabilidade do modelo.
cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores_base = cross_val_score(pipe_base, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1)

print(f"\n{'='*70}")
print(f" 📍 RESULTADOS BASELINE".center(70))
print(f"{'='*70}")

# =====================================================
# 4) Avaliação por validação cruzada (Treino)
# =====================================================

print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_base.mean():>15.4f} ± {cv_scores_base.std():.4f}")

# =====================================================
# 5) Avaliação no conjunto de teste
# =====================================================
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_base):>10.4f}")
print(f"   Otimizado:                 {max_acc_base:>10.4f} (threshold ={best_th_base:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_base):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_base):>10.4f}")

# =====================================================
# 6) Hiperparametros utilizados
# =====================================================
print(f"{'='*70}")
print_hiper(model_base.get_params())
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))


#Processo iniciado em: 17:16:44
# Iniciando treino rápido em 490000 linhas...
✅ Treino concluído!

                         📍 RESULTADOS BASELINE                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.6960 ± 0.0009

✅ TEST SET
   Padrão (0.5):                  0.6640
   Otimizado:                     0.6640 (threshold = 0.500)
   ROC-AUC:                       0.6948
   Avg precision:                 0.7897
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• C                         : 1.0
• Class Weight              : None
• Dual                      : False
• Fit Intercept             : True
• Intercept Scaling         : 1
• Loss                      : squared_hinge
• Max Iter                  : 2000
• Multi Class               : ovr
• Penalty                   : l2
• Random State              : 42
• Tol                       : 0.0001
• Verbose                   : 0
--------------------------------------------------

#Processo finali

In [15]:
print("#Processo iniciado em:", time.strftime("%H:%M:%S"))

param_grid_1 = {
    'model__estimator__C': [0.1, 0.5,1],
    'model__estimator__class_weight': [None, 'balanced'],
    'model__method': ['sigmoid', 'isotonic']
}

search_1 = GridSearchCV(
    pipe_base, 
    param_grid=param_grid_1,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=2
)

start = time.time()
search_1.fit(X_train, y_train)
print("✅ Treino concluído!")
end = time.time()

# -----------------------------------------------------
# 1) Predição
# -----------------------------------------------------
best_1 = search_1.best_estimator_
y_pred_1=best_1.predict(X_test)

# -----------------------------------------------------
# 2) Otimização do threshold de decisão
# -----------------------------------------------------
best_th_1,max_acc_1,y_probs_1=best_threshold2(best_1, X_train, y_train,X_test,y_test)

print("\n#Finalizando a validação cruzada em:", time.strftime("%H:%M:%S"))

# -----------------------------------------------------
# 3) Desempenho em validação cruzada
# -----------------------------------------------------
cv_scores_1 = cross_val_score(best_1, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1) #90s

print(f"\n{'='*70}")
print(f" 📍  GridSearchCV".center(70))
print(f"{'='*70}")

# -----------------------------------------------------
# 4) Avaliação por validação cruzada (Treino)
# -----------------------------------------------------
print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_1.mean():>15.4f} ± {cv_scores_1.std():.4f}")

# -----------------------------------------------------
# 5) Avaliação no conjunto de teste
# -----------------------------------------------------
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_1):>10.4f}")
print(f"   Otimizado:                 {max_acc_1:>10.4f} (threshold ={best_th_1:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_1):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_1):>10.4f}")

#-----------------------------------------------------
# 6) Hiperparametros utilizados
#-----------------------------------------------------
print(f"{'='*70}")
print_hiper(search_1.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

# 📊 CROSS-VALIDATION
#    Média roc_auc:                0.6962 ± 0.0009

# ✅ TEST SET
#    Padrão (0.5):                  0.6641
#    Otimizado:                     0.6641 (threshold = 0.500)
#    ROC-AUC:                       0.6951
#    Avg precision:                 0.7899
# ======================================================================
# 🎯 Melhores Hiperparâmetros
# --------------------------------------------------
# • Estimator  C              : 0.001
# • Estimator  Class Weight   : balanced
# • Method                    : sigmoid

#Processo iniciado em: 18:26:50
Fitting 3 folds for each of 12 candidates, totalling 36 fits
✅ Treino concluído!

#Finalizando a validação cruzada em: 18:29:04

                            📍  GridSearchCV                           
📊 CROSS-VALIDATION
   Média roc_auc:                0.6962 ± 0.0009

✅ TEST SET
   Padrão (0.5):                  0.6641
   Otimizado:                     0.6641 (threshold = 0.500)
   ROC-AUC:                       0.6951
   Avg precision:                 0.7899
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Estimator  C              : 0.1
• Estimator  Class Weight   : balanced
• Method                    : sigmoid
--------------------------------------------------

#Processo finalizado em: 18:29:21


In [17]:
joblib.dump(search_1, 'search_LinearSVC_final.joblib')
joblib.dump(search_1.best_estimator_, 'modelo_LinearSVC_final.'+mtd_scoring+'_v3a.joblib')

['modelo_LinearSVC_final.roc_auc_v3a.joblib']

[CV] END model__estimator__C=0.1, model__estimator__class_weight=None, model__method=isotonic; total time=  25.5s
[CV] END model__estimator__C=0.5, model__estimator__class_weight=None, model__method=isotonic; total time=  21.2s
[CV] END model__estimator__C=0.5, model__estimator__class_weight=balanced, model__method=isotonic; total time=  24.3s
[CV] END model__estimator__C=1, model__estimator__class_weight=None, model__method=isotonic; total time=  25.3s
[CV] END model__estimator__C=0.1, model__estimator__class_weight=None, model__method=sigmoid; total time=  21.9s
[CV] END model__estimator__C=0.1, model__estimator__class_weight=balanced, model__method=sigmoid; total time=  22.8s
[CV] END model__estimator__C=0.5, model__estimator__class_weight=None, model__method=isotonic; total time=  23.0s
[CV] END model__estimator__C=1, model__estimator__class_weight=None, model__method=sigmoid; total time=  20.5s
[CV] END model__estimator__C=1, model__estimator__class_weight=balanced, model__method=